# Train the oscilloscope detector on a free GPU

1. In Colab: **Runtime -> Change runtime type -> T4 GPU**.
2. Run the cells in order. When asked, upload `colab_dataset.zip` from the project folder.
3. The last cell downloads `best.pt`. Put it in the project's `models/` folder.

In [ ]:
!pip -q install ultralytics

In [ ]:
from google.colab import files
files.upload()  # choose colab_dataset.zip
!rm -rf yolo && unzip -q colab_dataset.zip
# point data.yaml at the Colab folder instead of the Windows path
import yaml
cfg = yaml.safe_load(open('yolo/data.yaml'))
cfg['path'] = '/content/yolo'
yaml.safe_dump(cfg, open('yolo/data.yaml', 'w'), sort_keys=False)
print(cfg)

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
model.train(data='yolo/data.yaml', epochs=100, imgsz=640, batch=16, patience=25,
            fliplr=0.0, seed=42, project='runs', name='detect', exist_ok=True)

In [ ]:
# Accuracy on the REAL test photos (never used for training)
best = YOLO('runs/detect/weights/best.pt')
m = best.val(data='yolo/data.yaml', split='test')
print(f'precision {m.box.mp:.3f}  recall {m.box.mr:.3f}  mAP50 {m.box.map50:.3f}  mAP50-95 {m.box.map:.3f}')

In [ ]:
files.download('runs/detect/weights/best.pt')